In [7]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent

from Source_Files.Model.Run_Model import load_env_variables_for_Local


def load_env_variables_for_Local():
    """
    .env 파일에서 환경 변수를 로드합니다.
    """
    # .env 파일의 경로를 지정합니다.
    # 현재 스크립트의 상대 경로로 설정합니다.
    # load_dotenv(dotenv_path='../../.env')
    # 절대 경로로 설정합니다.
    load_dotenv(dotenv_path='.env')

    os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
    os.environ['LANGSMITH_ENDPOINT'] = os.getenv('LANGSMITH_ENDPOINT')
    os.environ['LANGSMITH_PROJECT'] = os.getenv('LANGSMITH_PROJECT')
    os.environ['LANGSMITH_TRACING'] = os.getenv('LANGSMITH_TRACING')
    os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')



load_env_variables_for_Local()


In [9]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent

from Source_Files.Model.Run_Model import load_env_variables_for_Local

# 프로세스 실행을 위한 환경설정 및 파라미터 준비
# LangSmith 사이트에서 아래 내용을 복사해서 .env 파일에 입력
# LANGCHAIN_TRACING_V2=true
# LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
# LANGCHAIN_API_KEY=<your_api_key>
# LANGCHAIN_PROJECT="<your_project_name>"

# 1. 프롬프트 템플릿 준비
#    {tools} : 사용가능한 도구를 Agent 에게 알려주기 위해 필요한 파라미터 (필수)
#    {tool_names} : 다음 작업(Action)에 필요한 도구들을 표시하는 파라미터 (필수)
#    {agent_scratchpad} : Agent 가 수행해야 하는 작업의 내용을 표시 (필수)
prompt = ChatPromptTemplate.from_template(
    """당신은 유능한 기상학자입니다. 필요한 경우 다음 도구들을 사용할 수 있습니다:
    {tools}

    Use the following format:
        Thought: you should always think about what to do
        Action: the action to take, should be one of [{tool_names}]
        Action Input: the input to the action
        Observation: the result of the action
        ... (this Thought/Action/Action Input/Observation can repeat N times)
        Thought: I now know the final answer
        Final Answer: the final answer to the original input question
    ============================= Chat history =============================
    {chat_history}
    ================================ Human Message =================================
    Question: {question}
    ============================= Messages Placeholder =============================
    Thought: {agent_scratchpad}
    """
)

# 2. LLM 초기화
openai_api_key = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(
    temperature=0,
    max_tokens=2048,
    model_name="gpt-4o-mini",
    streaming=True
)

# 3. DuckDuckGo 검색 도구 초기화
search = DuckDuckGoSearchRun()
tools = [search]

# 4. agent 생성
agent = create_react_agent(llm, tools, prompt)
# 4-1. agent_executor 생성
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 이전 챗을 기억하기 위해 chat_history 사용
chat_history = []

# 5. 챗 실행
while True:
    # 5-1. 사용자 질문을 입력 받음
    user_input = input("\n\n당신 (종료 q): ")
    if user_input in ("끝", "종료", "q", "exit"):
        break

    # 5-2. Agent 실행
    result = agent_executor.invoke(
        {"question": user_input, "chat_history": chat_history}
    )
    # 5-3. 응답이 dictionary 로 전달되므로 output 만 추출해서 출력
    output_text = result["output"]
    print(f"\nAI --->\n{output_text}")

    # 대화내역 저장
    chat_history.append(f"user: {user_input}")
    chat_history.append(f"assistant: {output_text}")



> Entering new AgentExecutor chain...
Thought: 2024년 태풍 발생 상황에 대한 정보를 얻기 위해 검색해야 한다.
Action: duckduckgo_search
Action Input: "2024 typhoon occurrence situation"June was particularly bad for the archipelago; three typhoons—Butchoy, Carina, and Dindo—made landfall one after the other. On June 1 Typhoon Butchoy hit Northern Luzon; Carina then tore Eastern Visayas on June 12. Typhoon Dindo struck, compounding an already bad situation as the area began to assess the damage and begin rehabilitation ... On 25 October, STS Kristine exited the PAR and made landfall in Vietnam on 27 October. In the Philippines, STS Kristine was the 11th tropical cyclone in 2024, out of an annual average of 20 tropical cyclones that typically affect the country. Following STS Kristine, Super Typhoon Leon (international name Kong-Rey) entered the PAR on 26 October ... The 2024 typhoon season in the Philippines was unlike any in history, marked by an unprecedented onslaught of six typhoons within just 30 days. ..

KeyboardInterrupt: Interrupted by user

In [12]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain.agents.react.agent import create_react_agent
from langchain.chains.llm_math.base import LLMMathChain
from langchain.agents import Tool
from langchain.agents.agent import AgentExecutor


load_dotenv()

# 1. 프롬프트 템플릿 준비
prompt = ChatPromptTemplate.from_template(
    """
    Answer the following questions as best you can.
    You have access to the following tools:
    {tools}

    Use the following format:
    Question: the input question you must answer
    Thought: you should always think about what to do
    Action: the action to take, should be one of [{tool_names}]
    Action Input: the input to the action
    Observation: the result of the action
    ... (this Thought/Action/Action Input/Observation can repeat N times)
    Thought: I now know the final answer
    Final Answer: the final answer to the original input question

    Begin!

    ============================= Chat history =============================
    {chat_history}
    ================================ Human Message =================================
    Question: {question}
    ============================= Messages Placeholder =============================
    Thought:{agent_scratchpad}
    """
)

# 2. LLM 초기화
openai_api_key = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(
    temperature=0,
    max_tokens=2048,
    model_name="gpt-4o-mini",
    streaming=True
)

# 3. Agent 도구 초기화
# 3-1. DuckDuckGo 검색 도구 초기화
search_tool = Tool(
    name="Search", func=DuckDuckGoSearchRun().run, description="DuckDuckGo 웹 검색 도구"
)

# 3-2. Math 계산 도구 초기화
math_tool = Tool.from_function(
    name="Calculator",
    func=LLMMathChain.from_llm(llm=llm).run,
    description="""Useful for when you need to answer questions about math.
        This tool is only for math questions and nothing else.
        Only input math expressions.""",
)
# 3-3. 도구 리스트 생성
tools = [search_tool, math_tool]

# 4. agent 생성
agent = create_react_agent(llm, tools, prompt)
# 4-1. agent_executor 생성
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 이전 챗을 기억하기 위해 chat_history 사용
chat_history = []

# 5. 챗 실행
while True:
    # 5-1. 사용자 질문을 입력 받음
    user_input = input("\n\n당신 (종료 q): ")
    if user_input in ("끝", "q", "exit"):
        break

    # 5-2. Agent 실행
    result = agent_executor.invoke(
        {"question": user_input, "chat_history": chat_history}
    )
    # 5-3. 응답이 dictionary 로 전달되므로 output 만 추출해서 출력
    output_text = result["output"]
    print(f"\nAI --->\n{output_text}")

    # 5-4. 대화내역 저장
    chat_history.append(f"user: {user_input}")
    chat_history.append(f"assistant: {output_text}")



> Entering new AgentExecutor chain...
I need to gather information about the recent flooding in Busan. This requires a search for the latest news or reports on the topic. 

Action: Search  
Action Input: "Busan flooding news October 2023"  Over 200 flood-related emergency calls were reported, including submerged vehicles, backflowing manholes, and damaged commercial and residential properties. Flooding in rivers and low-lying areas prompted evacuations and led to the closure of major roads, bridges, and promenades. The rain also caused train suspensions between Dongdaegu and Busan. More than 110 reports of damage were received by the Busan Fire Department, most of which were related to flooding. Car damage is the biggest as all roads in the city are submerged. At 8:45 a.m. that day, a road in Yeonje-gu, Busan, was flooded with water, and five to six vehicles were reported. In Busan, 161 damage reports were received, of which 30 were reported for road and vehicle flooding and 20 were 

KeyboardInterrupt: Interrupted by user

In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1) 원래 베이스 모델 ID (예: 저번에 QLoRA 학습 전 양자화된 버전)
base_model_id = "Qwen/Qwen3-4B"

# 2) 업로드된(머지된) 모델 ID
merged_model_id = "SiniDSBA/qwen_qlora"

# 토크나이저도 같을 겁니다.
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 3) 모델 로드
#    만약 두 모델 모두 4bit 양자화된 상태라면, load_in_4bit 같은 옵션을 추가해야 할 수 있습니다.
base = AutoModelForCausalLM.from_pretrained(base_model_id, device_map="auto", load_in_4bit=True)
merged = AutoModelForCausalLM.from_pretrained(merged_model_id, device_map="auto", load_in_4bit=True)

# 4) state_dict 추출
base_sd   = base.state_dict()
merged_sd = merged.state_dict()

# 5) 파라미터가 하나라도 다른지 검사
diff_count = 0
for key in merged_sd:
    b = base_sd[key]
    m = merged_sd[key]
    # 4-bit 양자화 커널은 수치가 float가 아닐 수 있으니, 움직임 확인 정도만 예시로 쓰겠습니다.
    if not torch.allclose(b.float(), m.float(), atol=1e-3):
        diff_count += 1

print(f"변경된 파라미터 개수: {diff_count}개")
if diff_count > 0:
    print("→ 머지가 제대로 됐습니다. (적어도 하나 이상의 weight가 바뀌었음)")
else:
    print("→ 차이가 감지되지 않았습니다. 머지가 안 된 상태일 수 있습니다.")


PackageNotFoundError: No package metadata was found for bitsandbytes